# 02 — Pemodelan LSTM pada Data Baru (v2): 5 Strategi Penyeimbangan Kelas

Notebook ini menguji performa model LSTM unidirectional pada representasi teks `processed_text_v2` dari dataset `banjir_processed_v2.csv` ($n = 8.648$).

**5 Strategi Penyeimbangan yang Diuji**:
1. **Natural Baseline** (Tanpa Penyeimbangan)
2. **Class Weight (CW)** (Cost-Sensitive Learning)
3. **Random Oversampling (ROS)**
4. **Random Undersampling (RUS)**
5. **SMOTE** (Synthetic Minority Over-sampling)

Seluruh model dievaluasi secara adil pada **Data Uji Terkunci ($n = 1.730$)** (20% Stratified Split, Seed 42).


In [ ]:
import os
import sys
from pathlib import Path

def resolve_path(filename):
    """Cari file secara rekursif di /kaggle/input (Kaggle) atau kandidat lokal."""
    # 1. Rekursif cari di /kaggle/input (menangani semua variasi mount Kaggle)
    if Path("/kaggle/input").exists():
        for root, _dirs, files in os.walk("/kaggle/input"):
            if filename in files:
                found = os.path.join(root, filename)
                print(f"[resolve_path] Ditemukan di Kaggle: {found}")
                return found
    # 2. Kandidat lokal workstation
    candidates = [
        Path(f"Data/processed/{filename}"),
        Path(f"Data/simulated/{filename}"),
        Path(f"Data/raw/{filename}"),
        Path(f"Data/{filename}"),
        Path(f"kamus/{filename}"),
        Path(f"Output/predictions/{filename}"),
        Path(f"../Data/processed/{filename}"),
        Path(f"../Data/simulated/{filename}"),
        Path(f"../Data/raw/{filename}"),
        Path(f"../kamus/{filename}"),
        Path(filename),
    ]
    for p in candidates:
        if p.exists():
            print(f"[resolve_path] Ditemukan lokal: {p}")
            return str(p)
    return filename


In [ ]:
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, recall_score
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler

# Kunci seed deterministik
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Load Data Baru V2
csv_path = resolve_path('banjir_processed_v2.csv')
df = pd.read_csv(csv_path)
col_text = 'processed_text_v2'
col_label = 'label'

# 80:20 Split -> lalu 90:10 Train:Val (Total 72% Train, 8% Val, 20% Test)
tr_val, test_df = train_test_split(df, test_size=0.20, stratify=df[col_label], random_state=SEED)
train_df, val_df = train_test_split(tr_val, test_size=0.10, stratify=tr_val[col_label], random_state=SEED)

tokenizer = Tokenizer(num_words=10000, oov_token='<OOV>')
tokenizer.fit_on_texts(train_df[col_text].astype(str))

X_tr = pad_sequences(tokenizer.texts_to_sequences(train_df[col_text].astype(str)), maxlen=50, padding='post', truncating='post')
X_va = pad_sequences(tokenizer.texts_to_sequences(val_df[col_text].astype(str)), maxlen=50, padding='post', truncating='post')
X_te = pad_sequences(tokenizer.texts_to_sequences(test_df[col_text].astype(str)), maxlen=50, padding='post', truncating='post')

y_tr = train_df[col_label].values
y_va = val_df[col_label].values
y_te = test_df[col_label].values

print(f'Train size: {len(X_tr)} | Val size: {len(X_va)} | Test size: {len(X_te)}')
print('Distribusi Train:', pd.Series(y_tr).value_counts().sort_index().to_dict())
print('Distribusi Test :', pd.Series(y_te).value_counts().sort_index().to_dict())


In [ ]:
def build_lstm(vocab_size=10000, embedding_dim=128, units=64, dropout=0.2, max_len=50, lr=0.0002):
    model = Sequential([
        Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len),
        LSTM(units),
        Dropout(dropout),
        Dense(3, activation='softmax')
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

print('Model LSTM builder siap.')


In [ ]:
# Eksekusi 5 Varian Penyeimbangan Kelas pada LSTM
strategies = ['Baseline', 'Class Weight', 'Random Oversampling', 'Random Undersampling', 'SMOTE']
results = []

for strat in strategies:
    print('=' * 60)
    print(f'MELATIH LSTM DENGAN STRATEGI: {strat}')
    print('=' * 60)
    
    X_train_res, y_train_res = X_tr.copy(), y_tr.copy()
    cw_dict = None
    
    if strat == 'Class Weight':
        cw_vals = compute_class_weight('balanced', classes=np.unique(y_tr), y=y_tr)
        cw_dict = dict(enumerate(cw_vals))
        print(f'Class weights: {cw_dict}')
    elif strat == 'Random Oversampling':
        ros = RandomOverSampler(random_state=SEED)
        X_train_res, y_train_res = ros.fit_resample(X_tr, y_tr)
    elif strat == 'Random Undersampling':
        rus = RandomUnderSampler(random_state=SEED)
        X_train_res, y_train_res = rus.fit_resample(X_tr, y_tr)
    elif strat == 'SMOTE':
        smote = SMOTE(random_state=SEED)
        X_train_res, y_train_res = smote.fit_resample(X_tr, y_tr)
        X_train_res = np.round(X_train_res).astype('int32')
        
    print(f'Ukuran data latih setelah resampling: {len(X_train_res)} sampel')
    
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)
    np.random.seed(SEED)
    
    model = build_lstm()
    es = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    
    model.fit(
        X_train_res, y_train_res,
        validation_data=(X_va, y_va),
        epochs=20,
        batch_size=16,
        class_weight=cw_dict,
        callbacks=[es],
        verbose=0
    )
    
    preds = np.argmax(model.predict(X_te, verbose=0), axis=1)
    acc = accuracy_score(y_te, preds)
    f1_mac = f1_score(y_te, preds, average='macro', zero_division=0)
    rec_net = recall_score(y_te, preds, average=None, zero_division=0)[1]
    
    print(f'{strat} -> Test Accuracy: {acc*100:.2f}% | Macro F1: {f1_mac*100:.2f}% | Recall Netral: {rec_net*100:.2f}%')
    results.append({
        'Strategi': strat,
        'Accuracy (%)': round(acc * 100, 2),
        'Macro F1 (%)': round(f1_mac * 100, 2),
        'Recall Netral (%)': round(rec_net * 100, 2)
    })


In [ ]:
# Rangkuman Tabel Hasil Komparasi
df_empiris = pd.DataFrame(results)
print('=' * 65)
print('HASIL LENGKAP 5 STRATEGI BALANCING PADA LSTM (DATA BARU V2)')
print('=' * 65)
print(df_empiris.to_string(index=False))

out_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('Output/predictions')
out_dir.mkdir(parents=True, exist_ok=True)
df_empiris.to_csv(out_dir / 'results_lstm_empiris_balancing.csv', index=False)
print(f'Tersimpan di: {out_dir / "results_lstm_empiris_balancing.csv"}')
